# CH14 — Taller BFS y DFS

Ejercicios sobre grafos con la clase `Graph` de Goodrich, Tamassia & Goldwasser (cap. 14).

**Contenido**

0. Qué se entrega
1. Preparación — construir grafos para probar
2. Ejercicio 1 — Los vecinos de un nodo
3. Ejercicio 2 — `dfs` y `bfs`
4. Ejercicio 3 — Vértices alcanzables desde un nodo
5. Escribe tu solución en `ejercicios_bfs_dfs.py`
6. Verifica tu archivo

---
## 0. Qué se entrega

Un archivo **`ejercicios_bfs_dfs.py`** (con ese nombre exacto) en tu repositorio, en

```
student_work/ch14_graph_algorithms/ejercicios_bfs_dfs.py
```

que defina las funciones:

| Función | Recibe | Hace |
|---|---|---|
| `vecinos(g, v)` | un `Graph` y un `Vertex` de ese grafo | retorna una `list` con los **elementos** de los vecinos de `v` |
| `dfs(g, v, d)` | un `Graph`, un `Vertex` y un `dict` | **modifica** `d`: agrega cada vértice descubierto por DFS desde `v`, asociado a la arista que lo descubrió. No retorna nada |
| `bfs(g, v)` | un `Graph` y un `Vertex` de ese grafo | **retorna** un `dict` `{Vertex: Edge}` con cada vértice descubierto por BFS desde `v` y la arista que lo descubrió (`v` asociado a `None`) |
| `alcanzables(g, v)` | un `Graph` y un `Vertex` de ese grafo | retorna una `list` con los **elementos** de los vértices que se pueden alcanzar desde `v` (incluido `v`) |

**Reglas**

- Las funciones se califican con pruebas automáticas que **no** están en este notebook: prueba tus funciones con más grafos que los ejemplos.
- Usa solo la interfaz pública del grafo: `g.vertices()`, `g.edges()`, `g.incident_edges(v)`, `g.degree(v)`, `g.get_edge(u, v)`, `e.opposite(v)`, `e.endpoints()`, `v.element()`, etc.
- Las funciones **no** deben modificar el grafo, ni imprimir, ni pedir datos con `input()`.
- Si quieres dejar pruebas dentro del `.py`, ponlas debajo de `if __name__ == "__main__":` para que no se ejecuten al importar el archivo.
- Si necesitas importar algo del libro, hazlo con `from goodrich.ch14.graph import Graph` (y similares).

---
## 1. Preparación — construir grafos para probar

Importamos `Graph` de la librería del curso. Para no crear vértice por vértice, la función `construir` arma un grafo **no dirigido** a partir de una lista de aristas:

| Método | Qué hace |
|---|---|
| `g.insert_vertex(x)` | crea un vértice con elemento `x` y retorna su `Vertex` |
| `g.insert_edge(u, v)` | crea la arista entre los `Vertex` `u` y `v` |

`construir` retorna el grafo `g` y un diccionario `V` que va de la etiqueta al `Vertex`: así `V['A']` es el vértice con elemento `'A'`.

In [1]:
import sys
sys.path.append(r'C:\Users\fator\Desktop\Proyectos1\Estructura_Datos\2026_02')

In [2]:
from goodrich.ch14.graph import Graph

In [3]:
def construir(aristas, aislados=()):
    # Construye un Graph no dirigido. Retorna (g, V) con V: etiqueta -> Vertex.
    g = Graph()
    V = {}
    for u, v in aristas:
        for x in (u, v):
            if x not in V:
                V[x] = g.insert_vertex(x)
    for x in aislados:
        V[x] = g.insert_vertex(x)
    for u, v in aristas:
        g.insert_edge(V[u], V[v])
    return g, V

El grafo de los ejemplos (`F` no tiene aristas):

```
    A ----- B
    |       |
    |       |
    C ----- D ----- E        F
```

In [4]:
g, V = construir(
    [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D'), ('D', 'E')],
    aislados=['F'],
)

g.vertex_count(), g.edge_count()

(6, 5)

---
## 2. Ejercicio 1 — Los vecinos de un nodo

Dos vértices son **vecinos** (adyacentes) si hay una arista entre ellos.

Escribe `vecinos(g, v)` que retorne una **lista** con los **elementos** de los vecinos del vértice `v`. El orden de la lista no importa.

| `v` | `vecinos(g, v)` |
|---|---|
| `A` | `['B', 'C']` |
| `D` | `['B', 'C', 'E']` |
| `E` | `['D']` |
| `F` | `[]` |

*Pista:* `g.incident_edges(v)` recorre las aristas que tocan a `v`, y `e.opposite(v)` da el otro extremo de la arista `e`.

In [5]:
ejemplos_vecinos = [
    (V['A'], ['B', 'C']),
    (V['D'], ['B', 'C', 'E']),
    (V['E'], ['D']),
    (V['F'], []),
]

---
## 3. Ejercicio 2 — `dfs` y `bfs`

Las dos funciones guardan lo que descubren en un diccionario `{Vertex: Edge}`, como en el libro: a cada vértice descubierto le asocian la **arista por la que se llegó a él** (la arista del árbol). El vértice de inicio queda asociado a `None`. Si `d[w]` es la arista `e`, el **padre** de `w` es `e.opposite(w)`.

Los vecinos de cada vértice se exploran en el orden en que los entrega `g.incident_edges(u)`. Los vértices que no se pueden alcanzar desde el inicio **no** aparecen en el diccionario.

### `dfs(g, v, d)`

DFS (*depth-first search*) avanza **lo más profundo posible**: desde `v` va al primer vecino no descubierto, sigue desde ese vecino y solo cuando se queda sin vecinos nuevos se devuelve (*backtracking*) para probar el siguiente vecino de `v`.

- **Modifica** el diccionario `d` que recibe; no retorna nada.
- Quien la llama pone `v` en `d` antes de llamarla: `d = {v: None}` y luego `dfs(g, v, d)`.

*Pista:* es recursiva. Para cada arista `e` de `v`, si el otro extremo `w` no está en `d`, guarda `d[w] = e` y llama `dfs(g, w, d)`.

### `bfs(g, v)`

BFS (*breadth-first search*) recorre **por niveles**: primero `v`, luego todos sus vecinos, luego los vecinos de esos vecinos que aún no se han descubierto, y así sucesivamente.

- **Crea** el diccionario (con `v` asociado a `None`) y lo **retorna**.

*Pista:* guarda el nivel actual en una lista; para cada vértice del nivel, los vecinos no descubiertos van al diccionario y a la lista del siguiente nivel. Repite hasta que el nivel quede vacío.

### Ejemplos

Para leer los resultados, la función `padres` de abajo traduce `{Vertex: Edge}` a `{elemento: elemento del padre}`:

| inicio | `padres` después de `dfs` | `padres(bfs(g, inicio))` |
|---|---|---|
| `A` | `{'A': None, 'B': 'A', 'D': 'B', 'C': 'D', 'E': 'D'}` | `{'A': None, 'B': 'A', 'C': 'A', 'D': 'B', 'E': 'D'}` |
| `D` | `{'D': None, 'B': 'D', 'A': 'B', 'C': 'A', 'E': 'D'}` | `{'D': None, 'B': 'D', 'C': 'D', 'E': 'D', 'A': 'B'}` |
| `E` | `{'E': None, 'D': 'E', 'B': 'D', 'A': 'B', 'C': 'A'}` | `{'E': None, 'D': 'E', 'B': 'D', 'C': 'D', 'A': 'B'}` |
| `F` | `{'F': None}` | `{'F': None}` |

Desde `A`, BFS descubre a `C` desde `A` (está en el primer nivel); DFS lo descubre desde `D` porque bajó por `A -> B -> D` antes de volver.

In [6]:
def padres(d):
    # Traduce {Vertex: Edge} a {elemento: elemento del padre} para poder leerlo.
    return {w.element(): (e.opposite(w).element() if e is not None else None) for w, e in d.items()}


ejemplos_dfs = [
    (V['A'], {'A': None, 'B': 'A', 'D': 'B', 'C': 'D', 'E': 'D'}),
    (V['D'], {'D': None, 'B': 'D', 'A': 'B', 'C': 'A', 'E': 'D'}),
    (V['E'], {'E': None, 'D': 'E', 'B': 'D', 'A': 'B', 'C': 'A'}),
    (V['F'], {'F': None}),
]

ejemplos_bfs = [
    (V['A'], {'A': None, 'B': 'A', 'C': 'A', 'D': 'B', 'E': 'D'}),
    (V['D'], {'D': None, 'B': 'D', 'C': 'D', 'E': 'D', 'A': 'B'}),
    (V['E'], {'E': None, 'D': 'E', 'B': 'D', 'C': 'D', 'A': 'B'}),
    (V['F'], {'F': None}),
]

---
## 4. Ejercicio 3 — Vértices alcanzables desde un nodo

Un vértice `w` es **alcanzable** desde `v` si hay un camino de `v` a `w`. El mismo `v` cuenta como alcanzable.

Escribe `alcanzables(g, v)` que retorne una **lista** con los **elementos** de todos los vértices alcanzables desde `v`. El orden de la lista no importa.

| `v` | `alcanzables(g, v)` |
|---|---|
| `A` | `['A', 'B', 'C', 'D', 'E']` |
| `E` | `['A', 'B', 'C', 'D', 'E']` |
| `F` | `['F']` |

*Pista:* tu `bfs` o tu `dfs` del ejercicio 2 ya descubren exactamente esos vértices.

In [7]:
ejemplos_alcanzables = [
    (V['A'], ['A', 'B', 'C', 'D', 'E']),
    (V['E'], ['A', 'B', 'C', 'D', 'E']),
    (V['F'], ['F']),
]

---
## 5. Escribe tu solución en `ejercicios_bfs_dfs.py`

Completa las funciones en la celda de abajo y ejecútala. La primera línea, `%%writefile ejercicios_bfs_dfs.py`, **no** corre el código: guarda el contenido de la celda en el archivo `ejercicios_bfs_dfs.py`, en la misma carpeta de este notebook (`student_work/ch14_graph_algorithms/`). Ese archivo es el que se califica.

- No cambies el nombre del archivo ni el de las funciones.
- Cada vez que modifiques tu solución, vuelve a ejecutar esta celda para sobrescribir el archivo, y luego la celda de la sección 6 para verificarlo.
- No olvides hacer *commit* y *push* de `ejercicios_bfs_dfs.py` a tu repositorio.

In [8]:
%%writefile ejercicios_bfs_dfs.py
from goodrich.ch14.graph import Graph


def vecinos(g, v):
    """Retorna una lista con los elementos de los vecinos de v."""
    pass


def dfs(g, v, d):
    """DFS desde v. Agrega a d cada vertice descubierto -> arista que lo descubrio.

    v ya debe estar en d antes de la llamada (normalmente d = {v: None}). No retorna nada.
    """
    pass


def bfs(g, v):
    """BFS desde v. Retorna un dict vertice descubierto -> arista que lo descubrio (v -> None)."""
    pass


def alcanzables(g, v):
    """Retorna una lista con los elementos de los vertices alcanzables desde v (incluido v)."""
    pass


if __name__ == "__main__":
    # tus pruebas (opcional)
    pass

Overwriting ejercicios_bfs_dfs.py


---
## 6. Verifica tu archivo

Esta celda importa **tu** `ejercicios_bfs_dfs.py` (debe estar en la misma carpeta que este notebook) y lo corre contra los ejemplos de arriba. Pasar estos ejemplos **no** garantiza la nota: las pruebas de calificación usan más grafos.

In [9]:
import importlib
import ejercicios_bfs_dfs
importlib.reload(ejercicios_bfs_dfs)  # vuelve a leer el .py si lo cambiaste

for v, esperado in ejemplos_vecinos:
    obtenido = ejercicios_bfs_dfs.vecinos(g, v)
    ok = isinstance(obtenido, list) and sorted(obtenido) == sorted(esperado)
    print('OK   ' if ok else 'FALLA', f'vecinos({v.element()}):', obtenido, '| esperado:', esperado)

for s, esperado in ejemplos_dfs:
    d = {s: None}
    ejercicios_bfs_dfs.dfs(g, s, d)
    obtenido = padres(d)
    print('OK   ' if obtenido == esperado else 'FALLA', f'dfs({s.element()}):', obtenido, '| esperado:', esperado)

for s, esperado in ejemplos_bfs:
    d = ejercicios_bfs_dfs.bfs(g, s)
    obtenido = padres(d) if isinstance(d, dict) else d
    print('OK   ' if obtenido == esperado else 'FALLA', f'bfs({s.element()}):', obtenido, '| esperado:', esperado)

for v, esperado in ejemplos_alcanzables:
    obtenido = ejercicios_bfs_dfs.alcanzables(g, v)
    ok = isinstance(obtenido, list) and sorted(obtenido) == sorted(esperado)
    print('OK   ' if ok else 'FALLA', f'alcanzables({v.element()}):', obtenido, '| esperado:', esperado)

FALLA vecinos(A): None | esperado: ['B', 'C']
FALLA vecinos(D): None | esperado: ['B', 'C', 'E']
FALLA vecinos(E): None | esperado: ['D']
FALLA vecinos(F): None | esperado: []
FALLA dfs(A): {'A': None} | esperado: {'A': None, 'B': 'A', 'D': 'B', 'C': 'D', 'E': 'D'}
FALLA dfs(D): {'D': None} | esperado: {'D': None, 'B': 'D', 'A': 'B', 'C': 'A', 'E': 'D'}
FALLA dfs(E): {'E': None} | esperado: {'E': None, 'D': 'E', 'B': 'D', 'A': 'B', 'C': 'A'}
OK    dfs(F): {'F': None} | esperado: {'F': None}
FALLA bfs(A): None | esperado: {'A': None, 'B': 'A', 'C': 'A', 'D': 'B', 'E': 'D'}
FALLA bfs(D): None | esperado: {'D': None, 'B': 'D', 'C': 'D', 'E': 'D', 'A': 'B'}
FALLA bfs(E): None | esperado: {'E': None, 'D': 'E', 'B': 'D', 'C': 'D', 'A': 'B'}
FALLA bfs(F): None | esperado: {'F': None}
FALLA alcanzables(A): None | esperado: ['A', 'B', 'C', 'D', 'E']
FALLA alcanzables(E): None | esperado: ['A', 'B', 'C', 'D', 'E']
FALLA alcanzables(F): None | esperado: ['F']


## Para las calificaciones

In [10]:
from goodrich.ch14.graph import Graph


def construir(aristas, aislados=()):
    g = Graph()
    V = {}
    for u, v in aristas:
        for x in (u, v):
            if x not in V:
                V[x] = g.insert_vertex(x)
    for x in aislados:
        V[x] = g.insert_vertex(x)
    for u, v in aristas:
        g.insert_edge(V[u], V[v])
    return g, V


#    A ----- B
#    |       |
#    C ----- D ----- E        F
g1, V1 = construir(
    [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D'), ('D', 'E')],
    aislados=['F'],
)

#        2
#        |
#    1 - 0 - 3       (estrella, elementos enteros)
#        |
#        4
g2, V2 = construir([(0, 1), (0, 2), (0, 3), (0, 4)])

# a - b - c - a   y   x - y   (dos componentes)
g3, V3 = construir([('a', 'b'), ('b', 'c'), ('a', 'c'), ('x', 'y')])

# (nombre, grafo, vertice, esperado)  -- el orden no importa: comparar con sorted
casos_vecinos = [
    ('g1 vecinos de A', g1, V1['A'], ['B', 'C']),
    ('g1 vecinos de D', g1, V1['D'], ['B', 'C', 'E']),
    ('g1 vecinos de F (aislado)', g1, V1['F'], []),
    ('g2 vecinos del centro 0', g2, V2[0], [1, 2, 3, 4]),
    ('g2 vecinos de la hoja 3', g2, V2[3], [0]),
]

# (nombre, grafo, inicio, esperado)  -- d = {inicio: None}; dfs(g, inicio, d); comparar padres(d)
casos_dfs = [
    ('g1 DFS desde A', g1, V1['A'], {'A': None, 'B': 'A', 'D': 'B', 'C': 'D', 'E': 'D'}),
    ('g1 DFS desde D', g1, V1['D'], {'D': None, 'B': 'D', 'A': 'B', 'C': 'A', 'E': 'D'}),
    ('g1 DFS desde F (aislado)', g1, V1['F'], {'F': None}),
    ('g2 DFS desde la hoja 3', g2, V2[3], {3: None, 0: 3, 1: 0, 2: 0, 4: 0}),
]

# (nombre, grafo, inicio, esperado)  -- comparar padres(bfs(g, inicio))
casos_bfs = [
    ('g1 BFS desde A', g1, V1['A'], {'A': None, 'B': 'A', 'C': 'A', 'D': 'B', 'E': 'D'}),
    ('g1 BFS desde D', g1, V1['D'], {'D': None, 'B': 'D', 'C': 'D', 'E': 'D', 'A': 'B'}),
    ('g1 BFS desde F (aislado)', g1, V1['F'], {'F': None}),
    ('g2 BFS desde la hoja 3', g2, V2[3], {3: None, 0: 3, 1: 0, 2: 0, 4: 0}),
]

# (nombre, grafo, vertice, esperado)  -- el orden no importa: comparar con sorted
casos_alcanzables = [
    ('g1 alcanzables desde A', g1, V1['A'], ['A', 'B', 'C', 'D', 'E']),
    ('g1 alcanzables desde F (aislado)', g1, V1['F'], ['F']),
    ('g2 alcanzables desde la hoja 3', g2, V2[3], [0, 1, 2, 3, 4]),
    ('g3 alcanzables desde a', g3, V3['a'], ['a', 'b', 'c']),
    ('g3 alcanzables desde y', g3, V3['y'], ['x', 'y']),
]